# 01 — Data Validation

Loads the event panel from `DATA_PATH`, validates required columns, builds the
main event-level dataset for the `[-1, +1]` window, and saves the
sample-selection flow table to `outputs/`.

**Run first.** All downstream notebooks depend on this producing a valid `event` dataset.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))  # makes src/ importable

from src.config import DATA_PATH, OUT_DIR, PRE_DAY, POST_DAY, WINDOW_LABEL, REQUIRED_COLS
from src.data_utils import load_panel, build_event_level_dataset

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Main window : {WINDOW_LABEL}')
print(f'Data path   : {DATA_PATH.resolve()}')
print(f'Output dir  : {OUT_DIR.resolve()}')

Main window : [-1,1]
Data path   : /Users/roncheng/Documents/dev/projects/thesis/code/data/panel_stage5_V2_final.csv
Output dir  : /Users/roncheng/Documents/dev/projects/thesis/code/outputs


## 1. Load raw panel

In [2]:
df = load_panel(DATA_PATH)
print('Raw panel shape :', df.shape)
print('Unique events   :', df['event_id'].nunique())
print('Date range      :', df['event_date'].min(), '→', df['event_date'].max())
df.dtypes

Raw panel shape : (298243, 40)
Unique events   : 19000
Date range      : 2015-01-16 00:00:00 → 2025-09-12 00:00:00


secid                   int64
permno                float64
ticker                 object
pends                  object
event_date     datetime64[ns]
date           datetime64[ns]
rel_day                 int64
Dispersion            float64
exdate                 object
dte                   float64
LN_IMPSKEW            float64
LN_IMPVOL             float64
LN_PC_OI              float64
LN_PC_VLM             float64
iv_put                float64
delta_put             float64
oi_put                float64
strike_put            float64
iv_call               float64
delta_call            float64
oi_call               float64
strike_call           float64
LN_MKTCAP             float64
mktcap                float64
prc                   float64
ret                   float64
vol                   float64
cfacshr               float64
TOTALVAR              float64
LN_TOTALVAR           float64
saleq                 float64
atq                   float64
sic                   float64
LN_VIX    

## 2. Column completeness check

In [3]:
import pandas as pd
n = len(df)
completeness = pd.DataFrame({
    'column': REQUIRED_COLS,
    'present': [c in df.columns for c in REQUIRED_COLS],
    'pct_non_null': [
        round(df[c].notna().mean() * 100, 2) if c in df.columns else None
        for c in REQUIRED_COLS
    ],
})
completeness

,column,present,pct_non_null
0,secid,True,100.00
1,event_date,True,100.00
2,date,True,100.00
3,rel_day,True,100.00
4,LN_IMPSKEW,True,100.00
5,LN_IMPVOL,True,100.00
6,LN_VIX,True,100.00
7,LN_MRKCAP,True,99.99
8,ret,True,99.99
9,vol,True,99.99


## 3. Build event-level dataset and save sample flow

In [4]:
event, sample_flow = build_event_level_dataset(df, pre_day=PRE_DAY, post_day=POST_DAY)

sample_flow.to_csv(OUT_DIR / 'sample_selection_main_window.csv', index=False)

print('Event-level dataset shape:', event.shape)
print('Saved: sample_selection_main_window.csv')
sample_flow

Event-level dataset shape: (11468, 50)
Saved: sample_selection_main_window.csv


,step,N
0,Raw panel rows,298243
1,Raw unique events,19000
2,Events with rel_day = -1,14691
3,Events with rel_day = 1,13336
4,Events surviving pre/post merge,11468
5,Non-missing key vars,11468
6,Final event-level observations,11468


## 4. Relative-day coverage check

In [5]:
df.groupby('rel_day')['event_id'].nunique().rename('unique_events').to_frame()

,unique_events
rel_day,
-10,14774
-9,14907
-8,15114
-7,15148
-6,14868
-5,14644
-4,14534
-3,14731
-2,14762
